# Population Transfer Matrix Test

## What is tested

`get_transformed_thermal_rates`, `get_transformed_driven_rates` and
`get_transformed_decay_rates` provide the rates for the population kinetic matrix.
This notebook checks that the **complete kinetic matrix** (including its diagonal)
equals the population-population block extracted from the relaxation superoperator.

The tests use the same samples, four kinds of relaxation, and two basis combinations
as `dephasing_matrix_test(3).ipynb`.

## List of tests

**Simple tests** (`run_tests_simple`)

1. `test_population_single_context`: the kinetic matrix of single contexts.

**Difficult tests** (`run_tests_difficult`)

2. `test_population_concatenation`: the kinetic matrix of contexts combined with `mars.concat`.
3. `test_population_multiplication`: the kinetic matrix of contexts combined with `@`.


In [1]:
import os
import sys
import typing as tp

import torch

# Makes MaRs importable when the notebook is run from the repository without installation
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import mars
import mars.population.transform as transform
from mars import spin_model
from mars.population.contexts import Context

dtype = torch.float64
FIELD = 5.1  # T

## Helper functions

### Samples and bases

In [2]:
def create_sample(g_values: tuple, zfs_values: tuple) -> spin_model.SolidSample:
    """Create a powder sample of an S = 1 system with the given g-tensor and (D, E) in Hz."""
    system = spin_model.SpinSystem(
        electrons=[1.0],
        g_tensors=[spin_model.Interaction(g_values, dtype=dtype)],
        electron_electron=[(0, 0, spin_model.DEInteraction(zfs_values, dtype=dtype))],
        dtype=dtype,
    )
    return spin_model.SolidSample(base_spin_system=system, gauss=0.001, lorentz=0.001, mesh=(4, 4), dtype=dtype)


def create_samples() -> tuple[spin_model.SolidSample, spin_model.SolidSample]:
    """Create the two different samples used by all tests."""
    sample_1 = create_sample((2.02, 2.04, 2.06), (200e6, 50e6))
    sample_2 = create_sample((2.02, 2.14, 2.16), (200e6, 70e6))
    return sample_1, sample_2


def get_eigen_basis(sample: spin_model.SolidSample, field: float) -> torch.Tensor:
    """Return the eigenvectors of the sample Hamiltonian at the given field."""
    F, _, _, Gz = sample.get_hamiltonian_terms()
    hamiltonian = F + Gz * torch.tensor(field, dtype=dtype)
    _, vectors = torch.linalg.eigh(hamiltonian.unsqueeze(-3))
    return vectors


def direct_sum_batched(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Return the block-diagonal direct sum of two batches of square matrices."""
    n, m = a.shape[-1], b.shape[-1]
    result = torch.zeros(*a.shape[:-2], n + m, n + m, dtype=a.dtype, device=a.device)
    result[..., :n, :n] = a
    result[..., n:, n:] = b
    return result

### Contexts

In [3]:
def rates(values: list[list[float]]) -> torch.Tensor:
    """Convert a nested list of rates to a tensor."""
    return torch.tensor(values, dtype=dtype)


def contexts_thermal_only(sample_1, sample_2, basis_1, basis_2):
    """Create two contexts with thermal rates only."""
    context_1 = Context(
        basis=basis_1,
        sample=sample_1,
        init_populations=[0.5, 0.35, 0.15],
        decay_rates=[0.0, 0.0, 0.0],
        thermal_rates=rates([[0.0, 100.0, 0.0], [100.0, 0.0, 100.0], [0.0, 100.0, 0.0]]),
        dtype=dtype,
    )
    context_2 = Context(
        basis=basis_2,
        sample=sample_2,
        init_populations=[0.3, 0.4, 0.3],
        decay_rates=[0.0, 0.0, 0.0],
        thermal_rates=rates([[0.0, 100.0, 0.0], [100.0, 0.0, 200.0], [0.0, 200.0, 0.0]]),
        dtype=dtype,
    )
    return context_1, context_2


def contexts_explicit_dephasing(sample_1, sample_2, basis_1, basis_2):
    """Create two contexts with decay, thermal rates and explicit dephasing."""
    context_1 = Context(
        basis=basis_1,
        sample=sample_1,
        init_populations=[0.5, 0.25, 0.25],
        decay_rates=[1000.0, 200.3, 100.2],
        thermal_rates=rates([[0.0, 100.0, 0.0], [100.0, 0.0, 100.0], [0.0, 100.0, 0.0]]),
        dephasing=[1e4, 1e5, 1e6],
        dtype=dtype,
    )
    context_2 = Context(
        basis=basis_2,
        sample=sample_2,
        init_populations=[0.3, 0.4, 0.3],
        decay_rates=[111.0, 1234.0, 1232.0],
        thermal_rates=rates([[0.0, 100.0, 0.0], [100.0, 0.0, 200.0], [0.0, 200.0, 0.0]]),
        dephasing=[2e5, 1e4, 1e3],
        dtype=dtype,
    )
    return context_1, context_2


def contexts_decay_only(sample_1, sample_2, basis_1, basis_2):
    """Create two contexts with decay and zero thermal rates."""
    context_1 = Context(
        basis=basis_1,
        sample=sample_1,
        init_populations=[0.5, 0.35, 0.15],
        decay_rates=[0.0, 200.3, 100.2],
        thermal_rates=torch.zeros(3, 3, dtype=dtype),
        dtype=dtype,
    )
    context_2 = Context(
        basis=basis_2,
        sample=sample_2,
        init_populations=[0.3, 0.4, 0.3],
        decay_rates=[111.0, 1234.0, 1232.0],
        thermal_rates=torch.zeros(3, 3, dtype=dtype),
        dtype=dtype,
    )
    return context_1, context_2


def contexts_thermal_and_driven(sample_1, sample_2, basis_1, basis_2):
    """Create two contexts with thermal and driven rates."""
    context_1 = Context(
        basis=basis_1,
        sample=sample_1,
        init_populations=[0.5, 0.35, 0.15],
        decay_rates=[0.0, 200.3, 100.2],
        thermal_rates=rates([[0.0, 50.0, 0.0], [50.0, 0.0, 50.0], [0.0, 50.0, 0.0]]),
        driven_rates=rates([[0.0, 25.0, 0.0], [25.0, 0.0, 25.0], [0.0, 25.0, 0.0]]),
        dtype=dtype,
    )
    context_2 = Context(
        basis=basis_2,
        sample=sample_2,
        init_populations=[0.3, 0.4, 0.3],
        decay_rates=[0.0, 0.0, 0.0],
        thermal_rates=rates([[0.0, 50.0, 0.0], [50.0, 0.0, 100.0], [0.0, 100.0, 0.0]]),
        driven_rates=rates([[0.0, 25.0, 0.0], [25.0, 0.0, 50.0], [0.0, 0.0, 0.0]]),
        dtype=dtype,
    )
    return context_1, context_2


CONTEXT_CREATORS = [
    contexts_thermal_only,
    contexts_explicit_dephasing,
    contexts_decay_only,
    contexts_thermal_and_driven,
]


def context_pairs(sample_1, sample_2) -> tp.Iterator[tuple[Context, Context]]:
    """Yield every kind of context pair in two combinations of bases."""
    for creator in CONTEXT_CREATORS:
        yield creator(sample_1, sample_2, "xyz", "zeeman")
        yield creator(sample_1, sample_2, "zfs", "multiplet")

### Common check

In [4]:
def construct_kinetic_matrix(
    context: Context, vectors: torch.Tensor
) -> torch.Tensor:
    """Build the complete kinetic generator from thermal, driven and decay rates.

    The off-diagonal element K[i, j] is the rate j -> i. Its diagonal contains
    negative outgoing transition rates and irreversible population losses.
    """
    thermal = context.get_transformed_thermal_rates(vectors)
    driven = context.get_transformed_driven_rates(vectors)
    decay = context.get_transformed_decay_rates(vectors)

    sources = [rates for rates in (thermal, driven) if rates is not None]
    if sources:
        kinetic = sum(sources).clone().real
    else:
        N = vectors.shape[-1]
        kinetic = torch.zeros(
            *vectors.shape[:-2], N, N, dtype=vectors.real.dtype, device=vectors.device
        )

    kinetic.diagonal(dim1=-2, dim2=-1).zero_()
    outgoing = kinetic.sum(dim=-2)
    if decay is None:
        diagonal = -outgoing
    else:
        diagonal = -outgoing - decay
    kinetic.diagonal(dim1=-2, dim2=-1).copy_(diagonal)
    return kinetic


def check_population_matrix(
    context: Context, vectors: torch.Tensor, atol: float = 1e-6, rtol: float = 1e-5
) -> None:
    """Compare the full kinetic generator with the relaxation superoperator population block."""
    kinetic = construct_kinetic_matrix(context, vectors)

    # Reference only: use the superoperator independently of the kinetic calculation.
    thermal_superop = context.get_transformed_thermal_superop(vectors)
    driven_superop = context.get_transformed_driven_superop(vectors)
    sources = [s for s in (thermal_superop, driven_superop) if s is not None]
    if sources:
        expected = transform.extract_transition_matrix_from_superoperator(sum(sources))
    else:
        expected = torch.zeros_like(kinetic)

    assert kinetic.shape == expected.shape, (
        f"Kinetic and superoperator matrices have different shapes: "
        f"{kinetic.shape} and {expected.shape}"
    )

    # Pure loss is the negative column sum; the diagonal is tested as well.
    decay = context.get_transformed_decay_rates(vectors)
    expected_loss = torch.zeros_like(kinetic[..., 0, :]) if decay is None else decay
    assert torch.allclose(kinetic.sum(dim=-2), -expected_loss, atol=atol, rtol=rtol), (
        "Incorrect kinetic column sums (outgoing rates / decay). "
        f"Max difference: {(kinetic.sum(dim=-2) + expected_loss).abs().max().item():.2e}"
    )

    N = kinetic.shape[-1]
    mask = ~torch.eye(N, dtype=torch.bool, device=kinetic.device)
    assert torch.all(kinetic[..., mask] >= -atol), "Negative off-diagonal transition rate"

    assert torch.allclose(kinetic, expected, atol=atol, rtol=rtol), (
        "Kinetic matrix differs from the superoperator population block. "
        f"Max difference: {(kinetic - expected).abs().max().item():.2e}"
    )

## Tests

In [5]:
def test_population_single_context():
    """Check the complete kinetic matrix of single contexts."""
    sample_1, sample_2 = create_samples()
    vectors_1 = get_eigen_basis(sample_1, FIELD)
    vectors_2 = get_eigen_basis(sample_2, FIELD)

    for context_1, context_2 in context_pairs(sample_1, sample_2):
        check_population_matrix(context_1, vectors_1)
        check_population_matrix(context_2, vectors_2)
    print("Single context test passed (the kinetic matrix matches the superoperator population block)")


def test_population_concatenation():
    """Check the complete kinetic matrix of concatenated contexts."""
    sample_1, sample_2 = create_samples()
    vectors = direct_sum_batched(get_eigen_basis(sample_1, FIELD), get_eigen_basis(sample_2, FIELD))

    for context_1, context_2 in context_pairs(sample_1, sample_2):
        check_population_matrix(mars.concat([context_1, context_2]), vectors)
    print("Concatenation test passed (the kinetic matrix matches the superoperator population block)")


def test_population_multiplication():
    """Check the complete kinetic matrix of Kronecker-product contexts."""
    sample_1, sample_2 = create_samples()
    vectors = transform.batched_kron(get_eigen_basis(sample_1, FIELD), get_eigen_basis(sample_2, FIELD))

    for context_1, context_2 in context_pairs(sample_1, sample_2):
        check_population_matrix(context_1 @ context_2, vectors)
    print("Multiplication test passed (the kinetic matrix matches the superoperator population block)")

## Run the tests

In [6]:
def run_tests_simple():
    """Run the tests for single contexts."""
    test_population_single_context()
    print("\nThe simple population transfer matrix test passed.")


def run_tests_difficult():
    """Run the tests for combined contexts."""
    tests = [test_population_concatenation, test_population_multiplication]
    for test in tests:
        test()
    print(f"\nAll {len(tests)} difficult population transfer tests passed.")

In [7]:
run_tests_simple()

Single context test passed (the kinetic matrix matches the superoperator population block)

The simple population transfer matrix test passed.


In [8]:
run_tests_difficult()

D:\ITC\РНФ_Курганский_2024\pythonProject\MarS\mars\population\concatination.py:270: UserWarning: Contexts are combined by direct sum (block-diagonal composition). The subsystems evolve independently until transitions between them are added, e.g., mars.concat((context_1, context_2)) + context_exchange. Transitions between subsystems with different Hamiltonians are treated as incoherent population transfer; spectral (chemical) exchange is not described. This is valid only in the slow-exchange limit, when the transition rates between subsystems are much smaller than the differences between their resonance frequencies.
  warnings.warn(


Concatenation test passed (the kinetic matrix matches the superoperator population block)
Multiplication test passed (the kinetic matrix matches the superoperator population block)

All 2 difficult population transfer tests passed.


D:\ITC\РНФ_Курганский_2024\pythonProject\MarS\mars\population\concatination.py:270: UserWarning: Contexts are combined by direct sum (block-diagonal composition). The subsystems evolve independently until transitions between them are added, e.g., mars.concat((context_1, context_2)) + context_exchange. Transitions between subsystems with different Hamiltonians are treated as incoherent population transfer; spectral (chemical) exchange is not described. This is valid only in the slow-exchange limit, when the transition rates between subsystems are much smaller than the differences between their resonance frequencies.
  warnings.warn(
